# 🔨 Crack 3D Reconstruction + Analysis Report
Run each cell in order. GPU runtime recommended (Runtime → Change runtime type → T4 GPU).

**Outputs per run:**
- `3D Crack Mesh` — interactive GLB viewer
- `Analysis Report` — 6-panel image: original · depth map · crack mask · displacement heatmap · depth difference · severity classification

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
%pip install -q gradio open3d transformers pillow accelerate opencv-python matplotlib


In [ ]:
# ── Cell 2: Imports & model load ──────────────────────────────────────────────
import cv2
import numpy as np
import open3d as o3d
import gradio as gr
import torch
import matplotlib
matplotlib.use("Agg")  # non-interactive backend — required in Colab/headless
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from transformers import pipeline
from PIL import Image
from scipy.spatial import cKDTree
import tempfile
import os

if torch.cuda.is_available():
    device = 0
    print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = -1
    print("⚠️  No GPU found — running on CPU (will be slow)")

print("Loading depth model… (first run downloads ~1.3 GB)")
depth_pipe = pipeline(
    "depth-estimation",
    model="depth-anything/Depth-Anything-V2-Large-hf",
    device=device,
)
print("✅ Model ready.")


In [ ]:
# ── Cell 3: Helper functions ──────────────────────────────────────────────────

def classify_severity(max_depth_mm: float, crack_area_pct: float):
    """Return (label, hex_color) based on displacement depth and crack area."""
    score = (max_depth_mm / 1000.0) * 0.6 + (crack_area_pct / 100.0) * 0.4
    if score < 0.05:
        return "Hairline", "#4CAF50"
    elif score < 0.15:
        return "Minor",    "#8BC34A"
    elif score < 0.30:
        return "Moderate", "#FFC107"
    elif score < 0.50:
        return "Severe",   "#FF5722"
    else:
        return "Critical", "#B71C1C"


def build_report(original_rgb, base_depth, crack_mask_raw,
                 displacement_map, final_depth, crack_intensity):
    """
    Renders a 6-panel analysis figure and returns it as an HxWx3 uint8 array.

    Panels:
      [0] Original image
      [1] AI base depth map         (inferno colormap)
      [2] Crack binary mask         (gray)
      [3] Displacement heatmap      (jet — how deep each crack pixel is carved)
      [4] Depth difference  Δ       (plasma — final_depth minus base_depth)
      [5] Severity classification overlay
    """
    # ── Derived quantities ────────────────────────────────────────────────────
    depth_diff     = final_depth - base_depth
    crack_pixels   = (crack_mask_raw > 10).astype(np.uint8)
    crack_area_pct = crack_pixels.mean() * 100.0
    max_disp_mm    = float(displacement_map.max())
    mean_disp_mm   = (float(displacement_map[crack_pixels == 1].mean())
                      if crack_pixels.any() else 0.0)
    severity_label, _ = classify_severity(max_disp_mm, crack_area_pct)

    # ── Severity RGB overlay ──────────────────────────────────────────────────
    sev_cmap  = LinearSegmentedColormap.from_list(
        "crack_sev", ["#4CAF50", "#FFC107", "#FF5722", "#B71C1C"]
    )
    disp_norm = displacement_map / (displacement_map.max() + 1e-6)
    sev_rgba  = sev_cmap(disp_norm)                       # HxWx4 float [0,1]
    sev_rgb   = (sev_rgba[:, :, :3] * 255).astype(np.uint8)

    alpha       = (crack_pixels[:, :, None] * 0.75).astype(np.float32)
    bg_dim      = (original_rgb * 0.35).astype(np.uint8)
    sev_overlay = (sev_rgb * alpha + bg_dim * (1 - alpha)).astype(np.uint8)

    # ── Figure layout ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(
        2, 3, figsize=(18, 12),
        facecolor="#1a1a1a",
        gridspec_kw={"hspace": 0.38, "wspace": 0.12},
    )
    fig.patch.set_facecolor("#1a1a1a")

    panel_data = [
        (original_rgb,     "Original Image",          None,       False),
        (base_depth,       "AI Base Depth Map",        "inferno",  True),
        (crack_mask_raw,   "Crack Binary Mask",        "gray",     True),
        (displacement_map, "Displacement Heatmap",     "jet",      True),
        (depth_diff,       "Depth Difference (Δ)",     "plasma",   True),
        (sev_overlay,      "Severity Classification",  None,       False),
    ]

    for ax, (data, title, cmap, use_colorbar) in zip(axes.flat, panel_data):
        ax.set_facecolor("#1a1a1a")
        if cmap is not None:
            im = ax.imshow(data, cmap=cmap, interpolation="bilinear")
            if use_colorbar:
                cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                cb.ax.yaxis.set_tick_params(color="#cccccc", labelsize=8)
                plt.setp(cb.ax.yaxis.get_ticklabels(), color="#cccccc")
                cb.outline.set_edgecolor("#555555")
        else:
            ax.imshow(data, interpolation="bilinear")
        ax.set_title(title, color="#eeeeee", fontsize=13, fontweight="bold", pad=8)
        ax.axis("off")
        for spine in ax.spines.values():
            spine.set_edgecolor("#444444")
            spine.set_linewidth(0.8)
            spine.set_visible(True)

    # Stats annotation under severity panel
    ax_sev = axes[1, 2]
    stats_text = (
        f"Severity: {severity_label}   "
        f"Max Δ-depth: {max_disp_mm:.1f}   "
        f"Mean Δ-depth: {mean_disp_mm:.1f}   "
        f"Crack area: {crack_area_pct:.2f}%"
    )
    ax_sev.text(
        0.5, -0.08, stats_text,
        transform=ax_sev.transAxes,
        ha="center", va="top",
        fontsize=9, color="#dddddd",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#2a2a2a",
                  edgecolor="#555", linewidth=0.8),
    )

    # Severity legend
    legend_items = [
        mpatches.Patch(color="#4CAF50", label="Hairline"),
        mpatches.Patch(color="#8BC34A", label="Minor"),
        mpatches.Patch(color="#FFC107", label="Moderate"),
        mpatches.Patch(color="#FF5722", label="Severe"),
        mpatches.Patch(color="#B71C1C", label="Critical"),
    ]
    ax_sev.legend(
        handles=legend_items, loc="upper right",
        fontsize=8, framealpha=0.6,
        facecolor="#1a1a1a", edgecolor="#555", labelcolor="#eeeeee",
    )

    fig.suptitle(
        f"Crack Reconstruction Analysis  ·  Displacement Strength: {crack_intensity}"
        f"  ·  Overall: {severity_label}",
        color="#ffffff", fontsize=15, fontweight="bold", y=0.99,
    )

    fig.canvas.draw()
    w_px, h_px = fig.canvas.get_width_height()
    report_arr = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8).reshape(h_px, w_px, 3)
    plt.close(fig)
    return report_arr


print("✅ Helper functions defined.")


In [ ]:
# ── Cell 4: Main reconstruction function ─────────────────────────────────────

def reconstruct_crack(input_image, crack_intensity=50):
    """
    Returns (glb_path, report_image_array).
      glb_path     — textured mesh for gr.Model3D  (None if mesh build fails)
      report_image — HxWx3 uint8 numpy array for gr.Image
    """
    report_img = None   # ensure always defined for the except branch

    if input_image is None:
        return None, None

    try:
        # ── 0. Normalise channels ──────────────────────────────────────────
        if input_image.ndim == 2:
            input_image = np.stack([input_image] * 3, axis=-1)
        elif input_image.shape[2] == 4:
            input_image = input_image[:, :, :3]
        input_image = input_image.astype(np.uint8)

        # ── 1. Crack mask → displacement map ──────────────────────────────
        gray = cv2.cvtColor(input_image, cv2.COLOR_RGB2GRAY)
        _, mask_raw = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY_INV)
        mask_blur = cv2.GaussianBlur(mask_raw, (15, 15), 0).astype(np.float32)
        displacement_map = (mask_blur / 255.0) * float(crack_intensity)

        # ── 2. AI depth prediction ─────────────────────────────────────────
        pil = Image.fromarray(input_image)
        depth_raw  = depth_pipe(pil)["depth"]
        base_depth = np.array(depth_raw).astype(np.float32)

        # ── 3. Align sizes ────────────────────────────────────────────────
        if base_depth.shape != displacement_map.shape:
            dh, dw = base_depth.shape
            displacement_map = cv2.resize(displacement_map, (dw, dh), interpolation=cv2.INTER_LINEAR)
            mask_raw         = cv2.resize(mask_raw,         (dw, dh), interpolation=cv2.INTER_NEAREST)
            input_image      = cv2.resize(input_image,      (dw, dh), interpolation=cv2.INTER_LINEAR)

        # ── 4. Combine depth + crack carving ──────────────────────────────
        final_depth = base_depth + displacement_map
        dmin, dmax  = final_depth.min(), final_depth.max()
        depth_norm  = (final_depth - dmin) / (dmax - dmin + 1e-6)
        depth_uint  = (depth_norm * 1000).astype(np.uint16)

        # ── 5. Build analysis report (always, before expensive Poisson) ───
        report_img = build_report(
            original_rgb    = input_image,
            base_depth      = base_depth,
            crack_mask_raw  = mask_raw,
            displacement_map = displacement_map,
            final_depth     = final_depth,
            crack_intensity = crack_intensity,
        )
        print("✅ Analysis report generated.")

        # ── 6. RGBD → point cloud ─────────────────────────────────────────
        color_o3d = o3d.geometry.Image(np.ascontiguousarray(input_image))
        depth_o3d = o3d.geometry.Image(np.ascontiguousarray(depth_uint))

        rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
            color_o3d, depth_o3d,
            depth_scale=1000.0, depth_trunc=3.0,
            convert_rgb_to_intensity=False,
        )

        h, w = depth_uint.shape
        fx = fy = max(h, w)
        intrinsic = o3d.camera.PinholeCameraIntrinsic(w, h, fx, fy, w / 2.0, h / 2.0)

        pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)
        pcd.transform([[1, 0, 0, 0],
                       [0,-1, 0, 0],
                       [0, 0,-1, 0],
                       [0, 0, 0, 1]])

        # ── 7. Downsample + denoise ───────────────────────────────────────
        pcd = pcd.voxel_down_sample(voxel_size=0.005)

        if len(pcd.points) < 500:
            print(f"⚠️  Too few points ({len(pcd.points)}) — skipping 3D mesh.")
            return None, report_img

        pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
        pcd.estimate_normals(
            search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.02, max_nn=30)
        )
        pcd.orient_normals_towards_camera_location([0, 0, 0])

        # ── 8. Poisson surface reconstruction ────────────────────────────
        mesh, _ = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=8)
        bbox = pcd.get_axis_aligned_bounding_box()
        mesh = mesh.crop(bbox)

        # ── 9. Vertex color baking (vectorized) ───────────────────────────
        pc_points  = np.asarray(pcd.points)
        pc_colors  = np.asarray(pcd.colors)
        mesh_verts = np.asarray(mesh.vertices)
        tree       = cKDTree(pc_points)
        _, idx     = tree.query(mesh_verts, workers=-1)
        mesh.vertex_colors = o3d.utility.Vector3dVector(pc_colors[idx])

        # ── 10. Save GLB ──────────────────────────────────────────────────
        glb_path = os.path.join(tempfile.gettempdir(), "crack_reconstruction.glb")
        o3d.io.write_triangle_mesh(glb_path, mesh)
        print(f"✅ Mesh saved → {glb_path}  ({len(mesh.triangles):,} triangles)")

        return glb_path, report_img

    except Exception:
        import traceback
        traceback.print_exc()
        return None, report_img   # still return report if we have it


print("✅ reconstruct_crack() defined.")


In [ ]:
# ── Cell 5: Gradio UI ─────────────────────────────────────────────────────────

with gr.Blocks(title="Crack 3D Reconstruction", theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        "## 🪨 Crack 3D Reconstruction + Analysis\n"
        "Upload a crack photo → interactive 3D mesh **and** a 6-panel analysis report."
    )

    # ── Controls ──────────────────────────────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=1):
            input_img = gr.Image(type="numpy", label="📷  Crack Image")
            intensity = gr.Slider(
                minimum=10, maximum=150, value=50, step=5,
                label="Crack Displacement Strength",
                info="Controls how deeply cracks are carved into the depth field.",
            )
            run_btn = gr.Button("⚙️  Generate 3D Model + Report", variant="primary")
            gr.Markdown(
                "**Tips**\n"
                "- Use a well-lit, high-contrast crack photo\n"
                "- Lower values (10–40) suit hairline cracks; higher (80–150) suit deep structural cracks\n"
                "- The report is generated first — it appears even if the 3D mesh fails\n"
                "- First run downloads ~1.3 GB model weights"
            )

    # ── Tabbed outputs ────────────────────────────────────────────────────────
    with gr.Tabs():
        with gr.Tab("🧊  3D Crack Mesh"):
            output_model = gr.Model3D(
                label="Interactive 3D Mesh",
                clear_color=[0.15, 0.15, 0.15, 1.0],
            )
            gr.Markdown("_Left-drag to rotate · Scroll to zoom · Right-drag to pan_")

        with gr.Tab("📊  Analysis Report"):
            output_report = gr.Image(
                type="numpy",
                label="6-Panel Crack Analysis",
                show_download_button=True,
            )
            gr.Markdown(
                "**Panels (left→right, top→bottom)**\n"
                "1. Original image\n"
                "2. AI base depth map — warmer = farther from camera\n"
                "3. Binary crack mask\n"
                "4. Displacement heatmap — how deep each crack pixel is carved\n"
                "5. Depth difference Δ = final − base (pure crack contribution)\n"
                "6. Severity overlay — green→yellow→red→dark-red = hairline→critical\n\n"
                "Stats (max Δ-depth, mean Δ-depth, crack area %) appear beneath the severity panel."
            )

    run_btn.click(
        fn=reconstruct_crack,
        inputs=[input_img, intensity],
        outputs=[output_model, output_report],
    )

# share=True exposes a public tunnel URL — required in Colab
demo.launch(share=True)
